In [1]:
import os
# %env NX_CUGRAPH_AUTOCONFIG=True

import networkx as nx
import pandas as pd
from tqdm import tqdm
import json
import re
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# nx.config.backend_priority = ["cugraph"] # remeber to add this into python script


PATH_TO_CONSTANTS = "../"
with open(PATH_TO_CONSTANTS+'constants.json') as f:
    CONSTANTS = json.load(f)
    

In [5]:
DIR = f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/FILTER_DATASET/"

dataset_files = sorted([x for x in os.listdir(DIR) if 'json' in x and ('filtered' not in x and 'rejected' not in x )])

full_ds = []

for file in dataset_files:
    file_path = os.path.join(DIR, file)
    with open(file_path, 'r', encoding='utf-8') as f:
        ds = [json.loads(line) for line in f]  
        full_ds.extend(ds)  
        
DIR2 = f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/FILTER_DATASET_OPENBIOLLM/"

dataset_files = sorted([x for x in os.listdir(DIR2) if 'json' in x and ('filtered' not in x and 'rejected' not in x )])

full_ds2 = []

for file in dataset_files:
    file_path = os.path.join(DIR2, file)
    with open(file_path, 'r', encoding='utf-8') as f:
        ds = [json.loads(line) for line in f]  
        full_ds2.extend(ds)  

In [6]:
len(full_ds), len(full_ds2)

(264373, 50000)

In [11]:


ids = set([i['id'] for i in full_ds2])

full_ds = [i for i in full_ds if i['id'] in ids]

len(full_ds), len(full_ds2)

(50000, 50000)

In [18]:
full_ds[0]

{'id': 0,
 'nodes': ['EFFECT:46', 'COMPOUND:78', 'COMPOUND:669', 'COMPOUND:8321'],
 'edges': [[2, 0], [2, 1], [3, 0], [3, 1], [1, 0], [1, 2], [1, 3]],
 'graphlet_id': 7,
 'edge_labels': ['has_effect',
  'increase_efficacy',
  'has_effect',
  'increase_efficacy',
  'increase_effect',
  'increase_efficacy',
  'increase_efficacy'],
 'node_names': ['neuromuscular blockade',
  'Botulinum_toxin',
  'Tobramycin',
  'Magnesium_chloride'],
 'graphlet_text': [' - 0: neuromuscular blockade \n',
  ' - 1: Botulinum_toxin \n',
  ' - 2: Tobramycin \n',
  ' - 3: Magnesium_chloride \n'],
 'num_nodes': 4,
 'graphlet_index': 4,
 'text': '**Analysis of Graphlet**\n\n* **Question Node:** `0: neuromuscular blockade` (provides context for inquiry)\n* **Answer Node:** `1: Botulinum_toxin` (not explicitly mentioned in the question, but inferable)\n* **Hidden Nodes:**\n\t+ `2: Tobramycin` (enabling multi-hop reasoning regarding antibiotic interactions)\n\t+ `3: Magnesium_chloride` (enabling multi-hop reasoning 

In [12]:
import re

parsed_data = []
double_true_counter = 0
error = 0

filtered = []
rejected = []

for i  in range(len(full_ds)):

    temp_text = full_ds[i]['filtering_text'].replace("""\\\n""", """\\n""")
    matches = re.findall(r"\{.*?\}", temp_text, re.DOTALL)

    if matches:
        last_json_str = matches[-1]  # Get the last JSON block
        try:
            parsed_json = json.loads(last_json_str, strict = False)  # Parse JSON
            if 'valid_question' not in parsed_json or 'original_answer_valid' not in parsed_json:
                print(i, parsed_json.keys())
                
                parsed_data.append(False)
                error+=1
            

            else:
                if(parsed_json['valid_question'] and parsed_json['original_answer_valid']):
                    double_true_counter+=1
                    filtered.append(full_ds[i])
                else:
                    rejected.append(full_ds[i])

                parsed_data.append(parsed_json)

        except json.JSONDecodeError as e:
            print(i,e)
            error+=1

            
    else:
        print(i," No JSON found in the output.")
    

23 dict_keys(['from', 'to', 'relation'])
42 Expecting ',' delimiter: line 15 column 4 (char 595)
47 dict_keys(['accuracy', 'completeness'])
49 dict_keys(['entity1', 'relation', 'entity2'])
101 dict_keys(['type', 'entity', 'relation', 'treatment'])
105 dict_keys(['from', 'to', 'relation'])
114 Expecting ',' delimiter: line 16 column 4 (char 2033)
156 dict_keys(['question_reasoning', 'valid_question', 'my_answer', 'answer_reasoning'])
170 dict_keys(['accuracy', 'completeness', 'rationale'])
209 dict_keys(['entity1', 'relation', 'entity2'])
223 Expecting ',' delimiter: line 12 column 4 (char 731)
245 Expecting property name enclosed in double quotes: line 2 column 32 (char 33)
284 dict_keys(['alignment_with_biomedical_principles', 'completeness'])
286 Expecting ',' delimiter: line 15 column 4 (char 922)
295 dict_keys(['connection', 'validity', 'rationale'])
301 dict_keys(['accuracy', 'completeness'])
311 Expecting ',' delimiter: line 28 column 4 (char 2809)
317 dict_keys(['Botulinum toxin

In [24]:
import re

parsed_data2 = []
double_true_counter2 = 0
error2 = 0

filtered2 = []
rejected2 = []

for i  in range(len(full_ds2)):

    temp_text = full_ds2[i]['filtering_text'].replace("""\\\n""", """\\n""")
    matches = re.findall(r"\{.*?\}", temp_text, re.DOTALL)

    if matches:
        last_json_str = matches[-1]  # Get the last JSON block
        try:
            parsed_json = json.loads(last_json_str, strict = False)  # Parse JSON
            if 'valid_question' not in parsed_json or 'original_answer_valid' not in parsed_json:
                print(i, parsed_json.keys())
                
                parsed_data2.append(False)
                error2+=1
            

            else:
                if(parsed_json['valid_question'] and parsed_json['original_answer_valid']):
                    double_true_counter2+=1
                    filtered2.append(full_ds2[i])
                else:
                    rejected2.append(full_ds2[i])

                parsed_data2.append(parsed_json)

        except json.JSONDecodeError as e:
            print(i,e)
            error2+=1

            
    else:
        print(i," No JSON found in the output.")
    

363  No JSON found in the output.
3280  No JSON found in the output.
4128  No JSON found in the output.
4939  No JSON found in the output.
5012  No JSON found in the output.
5190 Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
5229  No JSON found in the output.
5458  No JSON found in the output.
5770  No JSON found in the output.
8745  No JSON found in the output.
9227  No JSON found in the output.
10460  No JSON found in the output.
11176  No JSON found in the output.
11237  No JSON found in the output.
11928 Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
13682  No JSON found in the output.
14053  No JSON found in the output.
20684  No JSON found in the output.
20808  No JSON found in the output.
22609  No JSON found in the output.
26724  No JSON found in the output.
26866  No JSON found in the output.
26882  No JSON found in the output.
28765  No JSON found in the output.
30641  No JSON found in the output.
33601  No JSON found

'Here is the evaluation in the requested JSON format:\n\n```\n{\n  "question_reasoning": "Analysis of the question reveals the following entities and connections: \n    * **Entities:** \n      1. Peginterferon_alfa-2a (drug)\n      2. PEG-uricase (enzyme)\n      3. Pegylated therapies (treatment approach)\n      4. Excessive uric acid production (condition)\n    * **Connections:**\n      1. Peginterferon_alfa-2a is related to the mechanism of action of the question\'s target enzyme.\n      2. The target enzyme is also utilized by other pegylated therapies.\n      3. The condition of focus is excessive uric acid production.\n  \n  **valid_question:** true,\n  \n  "my_answer": "The enzyme in question is indeed PEG-uricase. However, to clarify, Peginterferon_alfa-2a does not directly target PEG-uricase. Instead, PEG-uricase is specifically utilized by pegylated therapies (e.g., Pegvaliase) to manage hyperuricemia by catalyzing uric acid\'s oxidation into allantoin, thus reducing uric acid

In [17]:
print(double_true_counter, error)
print(double_true_counter/len(full_ds)*100, error/len(full_ds)*100, (len(rejected)/len(full_ds)*100))


print(double_true_counter2, error2)
print(double_true_counter2/len(full_ds)*100, error2/len(full_ds)*100, (len(rejected2)/len(full_ds)*100))

22196 3589
44.391999999999996 7.178 48.412
49931 3
99.862 0.006 0.06999999999999999


In [25]:
filtered2[0]

{'id': 0,
 'nodes': ['EFFECT:46', 'COMPOUND:78', 'COMPOUND:669', 'COMPOUND:8321'],
 'edges': [[2, 0], [2, 1], [3, 0], [3, 1], [1, 0], [1, 2], [1, 3]],
 'graphlet_id': 7,
 'edge_labels': ['has_effect',
  'increase_efficacy',
  'has_effect',
  'increase_efficacy',
  'increase_effect',
  'increase_efficacy',
  'increase_efficacy'],
 'node_names': ['neuromuscular blockade',
  'Botulinum_toxin',
  'Tobramycin',
  'Magnesium_chloride'],
 'graphlet_text': [' - 0: neuromuscular blockade \n',
  ' - 1: Botulinum_toxin \n',
  ' - 2: Tobramycin \n',
  ' - 3: Magnesium_chloride \n'],
 'num_nodes': 4,
 'graphlet_index': 4,
 'text': '**Analysis of Graphlet**\n\n* **Question Node:** `0: neuromuscular blockade` (provides context for inquiry)\n* **Answer Node:** `1: Botulinum_toxin` (not explicitly mentioned in the question, but inferable)\n* **Hidden Nodes:**\n\t+ `2: Tobramycin` (enabling multi-hop reasoning regarding antibiotic interactions)\n\t+ `3: Magnesium_chloride` (enabling multi-hop reasoning 

In [9]:
len(filtered), len(rejected), len(filtered)+len(rejected)

(119856, 127416, 247272)

In [27]:
# print(full_ds['264108']['filtering_text'])

with open(f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/FILTER_DATASET/filtered_ds.jsonl", 'w') as fp:
        for item in filtered:
            fp.write(json.dumps(item) + "\n")

with open(f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/FILTER_DATASET/rejected_ds.jsonl", 'w') as fp:
        for item in rejected:
            fp.write(json.dumps(item) + "\n")

In [12]:
from collections import Counter

counter_filtered = Counter(int(v['graphlet_id']) for v in filtered)
sorted_counts_filtered = dict(sorted(counter_filtered.items()))

# Count occurrences in full dataset
counter_full = Counter(int(v['graphlet_id']) for v in full_ds)
sorted_counts_full = dict(sorted(counter_full.items()))

# Compute ratio (filtered / full), rounded to 2 decimal places
ratios = {k: round((sorted_counts_filtered.get(k, 0) / sorted_counts_full[k])*100, 1) 
          for k in sorted_counts_full if k in sorted_counts_filtered}

print("Filtered Counts:", sorted_counts_filtered)
print("Full Dataset Counts:", sorted_counts_full)
print("Ratios:", ratios)

Filtered Counts: {1: 4544, 2: 1744, 3: 4149, 4: 4475, 5: 5325, 6: 4485, 7: 4365, 8: 5212, 9: 3485, 10: 3679, 11: 4390, 12: 4144, 13: 3897, 14: 4275, 15: 3628, 16: 5087, 17: 4459, 18: 2725, 19: 3606, 20: 5496, 21: 3281, 22: 4533, 23: 4629, 24: 4149, 25: 3759, 26: 3036, 27: 1647, 28: 5593, 29: 6059}
Full Dataset Counts: {1: 9913, 2: 3690, 3: 9783, 4: 10021, 5: 10103, 6: 9810, 7: 9870, 8: 9948, 9: 9817, 10: 9806, 11: 9939, 12: 9874, 13: 9885, 14: 9723, 15: 9823, 16: 9841, 17: 9949, 18: 6103, 19: 9878, 20: 9781, 21: 9629, 22: 9846, 23: 9621, 24: 9741, 25: 9781, 26: 5292, 27: 3690, 28: 9577, 29: 9639}
Ratios: {1: 45.8, 2: 47.3, 3: 42.4, 4: 44.7, 5: 52.7, 6: 45.7, 7: 44.2, 8: 52.4, 9: 35.5, 10: 37.5, 11: 44.2, 12: 42.0, 13: 39.4, 14: 44.0, 15: 36.9, 16: 51.7, 17: 44.8, 18: 44.7, 19: 36.5, 20: 56.2, 21: 34.1, 22: 46.0, 23: 48.1, 24: 42.6, 25: 38.4, 26: 57.4, 27: 44.6, 28: 58.4, 29: 62.9}


In [13]:
for k,v in sorted_counts_filtered.items():
    print(k, 
        #   sorted_counts_filtered[k], sorted_counts_full[k], 
          ratios[k]) 
    


1 45.8
2 47.3
3 42.4
4 44.7
5 52.7
6 45.7
7 44.2
8 52.4
9 35.5
10 37.5
11 44.2
12 42.0
13 39.4
14 44.0
15 36.9
16 51.7
17 44.8
18 44.7
19 36.5
20 56.2
21 34.1
22 46.0
23 48.1
24 42.6
25 38.4
26 57.4
27 44.6
28 58.4
29 62.9


In [14]:
with open(PATH_TO_CONSTANTS + CONSTANTS['templates_metadata']) as f:
    template_metadata = json.load(f)


In [15]:
template_metadata

{'1': {'counts': 2980635,
  'ratio': 0.0033549897924435566,
  'final_counts': 9954},
 '2': {'counts': 3702, 'ratio': 1.0, 'final_counts': 3702},
 '4': {'counts': 41964954,
  'ratio': 0.0002382940774818912,
  'final_counts': 10108},
 '3': {'counts': 50513861,
  'ratio': 0.00019796546536009197,
  'final_counts': 9826},
 '6': {'counts': 71664, 'ratio': 0.1395400759098013, 'final_counts': 9913},
 '5': {'counts': 3609661,
  'ratio': 0.0027703432538401804,
  'final_counts': 10165},
 '7': {'counts': 13537, 'ratio': 0.7387161113983896, 'final_counts': 9939},
 '8': {'counts': 11794, 'ratio': 0.8478887569950823, 'final_counts': 10038},
 '11': {'counts': 584613716,
  'ratio': 1.7105311980056247e-05,
  'final_counts': 10126},
 '10': {'counts': 1810874588,
  'ratio': 5.52219356672534e-06,
  'final_counts': 9952},
 '9': {'counts': 1080297928,
  'ratio': 9.256705711278564e-06,
  'final_counts': 9988},
 '14': {'counts': 871384, 'ratio': 0.011475996805082489, 'final_counts': 9946},
 '12': {'counts': 92

In [21]:
for k,v in sorted_counts_filtered.items():
    str_k = str(k)
    print(f"{k} & "
      f"{template_metadata[str_k]['counts']:,} & "
      f"${template_metadata[str_k]['ratio']:.2e}".replace("e-0", "e-").replace("e", " \\times 10^{") + "}$ & "
      f"{template_metadata[str_k]['final_counts']:,} & "
      f"{sorted_counts_full[k]:,} & "
      f"{sorted_counts_filtered[k]:,} & "
      f"{ratios[k]} \% \\\\")

    

1 & 2,980,635 & $3.35 \times 10^{-3}$ & 9,954 & 9,913 & 4,544 & 45.8 \% \\
2 & 3,702 & $1.00 \times 10^{+00}$ & 3,702 & 3,690 & 1,744 & 47.3 \% \\
3 & 50,513,861 & $1.98 \times 10^{-4}$ & 9,826 & 9,783 & 4,149 & 42.4 \% \\
4 & 41,964,954 & $2.38 \times 10^{-4}$ & 10,108 & 10,021 & 4,475 & 44.7 \% \\
5 & 3,609,661 & $2.77 \times 10^{-3}$ & 10,165 & 10,103 & 5,325 & 52.7 \% \\
6 & 71,664 & $1.40 \times 10^{-1}$ & 9,913 & 9,810 & 4,485 & 45.7 \% \\
7 & 13,537 & $7.39 \times 10^{-1}$ & 9,939 & 9,870 & 4,365 & 44.2 \% \\
8 & 11,794 & $8.48 \times 10^{-1}$ & 10,038 & 9,948 & 5,212 & 52.4 \% \\
9 & 1,080,297,928 & $9.26 \times 10^{-6}$ & 9,988 & 9,817 & 3,485 & 35.5 \% \\
10 & 1,810,874,588 & $5.52 \times 10^{-6}$ & 9,952 & 9,806 & 3,679 & 37.5 \% \\
11 & 584,613,716 & $1.71 \times 10^{-5}$ & 10,126 & 9,939 & 4,390 & 44.2 \% \\
12 & 922,997 & $1.08 \times 10^{-2}$ & 10,078 & 9,874 & 4,144 & 42.0 \% \\
13 & 772,905 & $1.29 \times 10^{-2}$ & 10,100 & 9,885 & 3,897 & 39.4 \% \\
14 & 871,384 & $1